# TF-IDF + Metadata + Author Priors + QWK Ensemble

This notebook is a non-transformer companion to the SciBERT notebooks. The goal is to create a different error profile using sparse text features and strong metadata features, then optimize Quadratic Weighted Kappa (QWK) with out-of-fold predictions.

Main ideas:

- word and character TF-IDF from titles,
- metadata text from venue, year, DOI type, and authors,
- explicit numeric and categorical metadata features,
- leakage-safe author target-prior features inside CV,
- regression/classification models blended on OOF QWK,
- threshold optimization for the final ordinal labels.

In [ ]:
# If running in Colab from a fresh runtime, uncomment and adapt this block.
# !git clone https://github.com/PhatLavar/DATA_MINING_ASSIGNMENT.git
# %cd DATA_MINING_ASSIGNMENT

from pathlib import Path
import os
import re
import json
import math
import random
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from scipy import sparse
from scipy.optimize import minimize

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge, LogisticRegression, HuberRegressor
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVR

warnings.filterwarnings('ignore')

SEED = 42
N_SPLITS = 5
LABEL_COL = 'Label'

random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'train.csv').exists() and (PROJECT_ROOT.parent / 'data' / 'train.csv').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
SUBMISSION_DIR = PROJECT_ROOT / 'submissions'
SUBMISSION_DIR.mkdir(exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Data dir:', DATA_DIR)
print('Submission dir:', SUBMISSION_DIR)

## Load Data

In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
public_test = pd.read_csv(DATA_DIR / 'public_test.csv')
private_test = pd.read_csv(DATA_DIR / 'private_test.csv')

test = pd.concat([
    public_test.assign(_split='public'),
    private_test.assign(_split='private')
], axis=0, ignore_index=True)

print('train:', train.shape)
print('public:', public_test.shape)
print('private:', private_test.shape)
print(train.head())

In [ ]:
print('Label distribution')
display(train[LABEL_COL].value_counts().sort_index().to_frame('count'))

print('\nVenue distribution by label')
display(pd.crosstab(train['venue'], train[LABEL_COL], normalize='index').round(3))

print('\nTrain venues:', train['venue'].value_counts().to_dict())
print('Test venues:', test['venue'].value_counts().to_dict())

print('\nYear by split')
display(pd.DataFrame({
    'train': train['year'].value_counts().sort_index(),
    'test': test['year'].value_counts().sort_index(),
}).fillna(0).astype(int))

## Metric and Threshold Utilities

We train models as continuous scorers, then convert continuous scores into labels with thresholds optimized directly for QWK.

In [ ]:
def qwk(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')


def apply_thresholds(pred, thresholds):
    thresholds = np.sort(np.asarray(thresholds, dtype=float))
    return np.digitize(pred, thresholds) + 1


def optimize_thresholds(y_true, pred, initial=None, verbose=False):
    y_true = np.asarray(y_true, dtype=int)
    pred = np.asarray(pred, dtype=float)
    if initial is None:
        initial = np.array([1.5, 2.5, 3.5, 4.5], dtype=float)

    def loss(thresholds):
        thresholds = np.sort(thresholds)
        labels = apply_thresholds(pred, thresholds)
        return -qwk(y_true, labels)

    result = minimize(
        loss,
        x0=np.asarray(initial, dtype=float),
        method='Nelder-Mead',
        options={'maxiter': 3000, 'xatol': 1e-7, 'fatol': 1e-7},
    )
    thresholds = np.sort(result.x)
    score = -result.fun
    if verbose:
        print('thresholds:', thresholds, 'qwk:', score)
    return thresholds, score


def print_score_block(name, y_true, pred):
    thresholds, score = optimize_thresholds(y_true, pred)
    labels = apply_thresholds(pred, thresholds)
    print(f'{name} QWK: {score:.6f}')
    print('thresholds:', np.round(thresholds, 5))
    print('distribution:', pd.Series(labels).value_counts().sort_index().to_dict())
    return thresholds, score, labels

## Feature Engineering

The author-prior features are built carefully inside each fold:

- validation/test rows only see author statistics from the training side of that fold,
- training rows use leave-one-row-out author statistics so a row does not directly leak its own label into its features.

In [ ]:
def clean_text(x):
    if pd.isna(x):
        return ''
    x = str(x).replace('&apos;', "'").replace('&quot;', '"')
    x = re.sub(r'\s+', ' ', x)
    return x.strip()


def doi_type(x):
    x = clean_text(x).lower()
    if 'semanticscholar' in x:
        return 'semanticscholar'
    if 'ceur-ws' in x:
        return 'ceur'
    if x.startswith('10.') or 'doi.org' in x:
        return 'doi'
    return 'other'


def split_authors(x):
    x = clean_text(x)
    if not x:
        return []
    parts = [p.strip().lower() for p in x.split(',')]
    return [p for p in parts if p]


KEYWORDS = [
    'proceedings', 'conference', 'workshop', 'symposium', 'invited',
    'answer set', 'asp', 'logic programming', 'neural', 'deep',
    'explainable', 'probabilistic', 'argumentation', 'planning',
    'verification', 'synthesis', 'model checking', 'temporal',
    'automata', 'constraint', 'optimization', 'knowledge representation'
]


def build_author_maps(df, y):
    sums = defaultdict(float)
    counts = defaultdict(int)
    for authors, label in zip(df['authors'], y):
        for author in split_authors(authors):
            sums[author] += float(label)
            counts[author] += 1
    return sums, counts


def author_prior_features(apply_df, fit_df, fit_y, loo=False):
    global_mean = float(np.mean(fit_y))
    sums, counts = build_author_maps(fit_df, fit_y)

    rows = []
    fit_labels_by_index = None
    if loo:
        fit_labels_by_index = pd.Series(fit_y, index=fit_df.index).to_dict()

    for idx, authors_raw in apply_df['authors'].items():
        authors = split_authors(authors_raw)
        values = []
        known = 0

        for author in authors:
            s = sums.get(author, 0.0)
            c = counts.get(author, 0)
            if loo and idx in fit_df.index and c > 0:
                s -= float(fit_labels_by_index[idx])
                c -= 1
            if c > 0:
                known += 1
                values.append(s / c)

        if values:
            mean_prior = float(np.mean(values))
            max_prior = float(np.max(values))
            min_prior = float(np.min(values))
            std_prior = float(np.std(values))
        else:
            mean_prior = global_mean
            max_prior = global_mean
            min_prior = global_mean
            std_prior = 0.0

        rows.append({
            'author_count': len(authors),
            'known_author_count': known,
            'author_prior_mean': mean_prior,
            'author_prior_max': max_prior,
            'author_prior_min': min_prior,
            'author_prior_std': std_prior,
        })

    return pd.DataFrame(rows, index=apply_df.index)


def make_base_features(df):
    out = pd.DataFrame(index=df.index)
    title = df['title'].map(clean_text)
    authors = df['authors'].map(clean_text)
    venue = df['venue'].map(clean_text).str.lower()
    year = df['year'].astype(str).map(clean_text)
    dtype = df['doi'].map(doi_type)

    out['title_text'] = title
    out['authors_text'] = authors.str.lower()
    out['meta_text'] = (
        'venue_' + venue +
        ' year_' + year +
        ' doi_' + dtype +
        ' authors ' + out['authors_text']
    )

    out['venue'] = venue
    out['year_str'] = year
    out['venue_year'] = venue + '_' + year
    out['doi_type'] = dtype

    out['year_num'] = pd.to_numeric(df['year'], errors='coerce').fillna(df['year'].median()).astype(float)
    out['title_len'] = title.str.len().astype(float)
    out['title_word_count'] = title.str.split().map(len).astype(float)
    out['has_authors'] = (authors.str.len() > 0).astype(float)
    out['doi_len'] = df['doi'].map(clean_text).str.len().astype(float)

    lower_title = title.str.lower()
    for kw in KEYWORDS:
        safe = re.sub(r'[^a-z0-9]+', '_', kw).strip('_')
        out[f'kw_{safe}'] = lower_title.str.contains(re.escape(kw), regex=True).astype(float)

    return out


def make_features(apply_df, fit_df, fit_y, loo=False):
    base = make_base_features(apply_df)
    author_feats = author_prior_features(apply_df, fit_df, fit_y, loo=loo)
    return pd.concat([base, author_feats], axis=1)

## Matrix Builder

Every fold fits its own vectorizers and encoders on the fold-training data only.

In [ ]:
TEXT_FEATURES = ['title_text', 'authors_text', 'meta_text']
CAT_FEATURES = ['venue', 'year_str', 'venue_year', 'doi_type']
NUM_FEATURES = [
    'year_num', 'title_len', 'title_word_count', 'has_authors', 'doi_len',
    'author_count', 'known_author_count', 'author_prior_mean',
    'author_prior_max', 'author_prior_min', 'author_prior_std',
] + [f'kw_{re.sub(r"[^a-z0-9]+", "_", kw).strip("_")}' for kw in KEYWORDS]


def make_preprocessor():
    try:
        onehot = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
    except TypeError:
        onehot = OneHotEncoder(handle_unknown='ignore', sparse=True)

    return ColumnTransformer(
        transformers=[
            ('title_word', TfidfVectorizer(
                lowercase=True,
                strip_accents='unicode',
                ngram_range=(1, 3),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True,
                max_features=25000,
            ), 'title_text'),
            ('title_char', TfidfVectorizer(
                lowercase=True,
                strip_accents='unicode',
                analyzer='char_wb',
                ngram_range=(3, 6),
                min_df=2,
                sublinear_tf=True,
                max_features=35000,
            ), 'title_text'),
            ('authors_word', TfidfVectorizer(
                lowercase=True,
                strip_accents='unicode',
                ngram_range=(1, 2),
                min_df=2,
                sublinear_tf=True,
                max_features=12000,
            ), 'authors_text'),
            ('meta_word', TfidfVectorizer(
                lowercase=True,
                strip_accents='unicode',
                token_pattern=r'(?u)\b\w[\w_]+\b',
                ngram_range=(1, 2),
                min_df=1,
                sublinear_tf=True,
                max_features=8000,
            ), 'meta_text'),
            ('cat', onehot, CAT_FEATURES),
            ('num', StandardScaler(with_mean=False), NUM_FEATURES),
        ],
        sparse_threshold=0.3,
        remainder='drop',
        verbose_feature_names_out=False,
    )

## Cross-Validated Models

We train several simple but strong models. The blend step decides how much each model should contribute.

In [ ]:
y = train[LABEL_COL].astype(int).values

models = {
    'ridge_6': Ridge(alpha=6.0, random_state=SEED),
    'ridge_15': Ridge(alpha=15.0, random_state=SEED),
    'linearsvr': LinearSVR(C=0.35, epsilon=0.05, loss='squared_epsilon_insensitive', random_state=SEED, max_iter=5000),
    'logreg_expected': LogisticRegression(C=1.0, class_weight='balanced', solver='liblinear', multi_class='ovr', random_state=SEED, max_iter=2000),
}

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_preds = {name: np.zeros(len(train), dtype=float) for name in models}
test_preds = {name: np.zeros(len(test), dtype=float) for name in models}
fold_scores = {name: [] for name in models}

for fold, (tr_idx, va_idx) in enumerate(skf.split(train, y), 1):
    print(f'\n========== Fold {fold}/{N_SPLITS} ==========', flush=True)
    train_fold = train.iloc[tr_idx].copy()
    valid_fold = train.iloc[va_idx].copy()
    y_train = y[tr_idx]
    y_valid = y[va_idx]

    X_train_df = make_features(train_fold, train_fold, y_train, loo=True)
    X_valid_df = make_features(valid_fold, train_fold, y_train, loo=False)
    X_test_df = make_features(test.copy(), train_fold, y_train, loo=False)

    preprocessor = make_preprocessor()
    X_train = preprocessor.fit_transform(X_train_df)
    X_valid = preprocessor.transform(X_valid_df)
    X_test = preprocessor.transform(X_test_df)

    print('matrix:', X_train.shape, 'nnz:', getattr(X_train, 'nnz', 'dense'))

    for name, model in models.items():
        model = clone(model)
        model.fit(X_train, y_train)

        if name == 'logreg_expected':
            valid_proba = model.predict_proba(X_valid)
            test_proba = model.predict_proba(X_test)
            classes = model.classes_.astype(float)
            valid_pred = valid_proba @ classes
            test_pred = test_proba @ classes
        else:
            valid_pred = model.predict(X_valid)
            test_pred = model.predict(X_test)

        valid_pred = np.clip(valid_pred, 1.0, 5.0)
        test_pred = np.clip(test_pred, 1.0, 5.0)

        oof_preds[name][va_idx] = valid_pred
        test_preds[name] += test_pred / N_SPLITS

        thresholds, score = optimize_thresholds(y_valid, valid_pred)
        fold_scores[name].append(score)
        print(f'{name:16s} fold QWK={score:.6f} thresholds={np.round(thresholds, 3)}')

In [ ]:
print('OOF results')
model_thresholds = {}
model_oof_scores = {}

for name, pred in oof_preds.items():
    thresholds, score, labels = print_score_block(name, y, pred)
    model_thresholds[name] = thresholds
    model_oof_scores[name] = score
    print('mean fold QWK:', np.mean(fold_scores[name]))
    print()

## Blend Search

This grid is intentionally small and robust. With only 2494 training rows, very fine blend optimization can overfit the OOF folds.

In [ ]:
def normalize_weights(w):
    w = np.asarray(w, dtype=float)
    s = w.sum()
    if s <= 0:
        return w
    return w / s

model_names = list(models.keys())
pair_weights = np.arange(0.0, 1.01, 0.1)

best = {
    'score': -999,
    'weights': None,
    'thresholds': None,
    'pred': None,
}

# Fast blend search: singles, pairwise blends, equal blend, and score-weighted blend.
# This avoids thousands of expensive threshold optimizations.
candidate_weights = []

# Single models.
for i in range(len(model_names)):
    w = np.zeros(len(model_names), dtype=float)
    w[i] = 1.0
    candidate_weights.append(w)

# Pairwise blends.
for i in range(len(model_names)):
    for j in range(i + 1, len(model_names)):
        for a in pair_weights:
            w = np.zeros(len(model_names), dtype=float)
            w[i] = a
            w[j] = 1.0 - a
            candidate_weights.append(w)

# Equal-weight and OOF-score-weighted blends.
candidate_weights.append(np.ones(len(model_names), dtype=float) / len(model_names))
score_w = np.array([max(model_oof_scores.get(name, 0.0), 0.0) for name in model_names], dtype=float)
if score_w.sum() > 0:
    candidate_weights.append(normalize_weights(score_w))

# De-duplicate rounded candidates.
unique = {}
for w in candidate_weights:
    w = normalize_weights(w)
    unique[tuple(np.round(w, 6))] = w
candidate_weights = list(unique.values())
print('Blend candidates:', len(candidate_weights))

for w in candidate_weights:
    blend = sum(w[i] * oof_preds[name] for i, name in enumerate(model_names))
    thresholds, score = optimize_thresholds(y, blend)
    if score > best['score']:
        best.update({
            'score': score,
            'weights': w,
            'thresholds': thresholds,
            'pred': blend,
        })

print('Best blend QWK:', best['score'])
print('Weights:')
for name, weight in zip(model_names, best['weights']):
    print(f'  {name:16s} {weight:.3f}')
print('Thresholds:', np.round(best['thresholds'], 5))

blend_oof_labels = apply_thresholds(best['pred'], best['thresholds'])
print('Blend OOF distribution:', pd.Series(blend_oof_labels).value_counts().sort_index().to_dict())
print('Confusion matrix:')
display(pd.DataFrame(confusion_matrix(y, blend_oof_labels, labels=[1,2,3,4,5]), index=[1,2,3,4,5], columns=[1,2,3,4,5]))

## Optional: Test-Venue Calibration Check

The test set has no `iclp`, so this cell also evaluates thresholds optimized only on training rows whose venues appear in test. If this score is close to global OOF, the venue-filtered thresholds may generalize better to leaderboard data.

In [ ]:
test_venues = set(test['venue'].astype(str).str.lower())
mask_test_venues = train['venue'].astype(str).str.lower().isin(test_venues).values

venue_thresholds, venue_score = optimize_thresholds(
    y[mask_test_venues],
    best['pred'][mask_test_venues],
)
venue_labels_all = apply_thresholds(best['pred'], venue_thresholds)
venue_score_all = qwk(y, venue_labels_all)

print('Global thresholds:', np.round(best['thresholds'], 5), 'OOF QWK:', best['score'])
print('Test-venue thresholds:', np.round(venue_thresholds, 5))
print('QWK on test-venue train rows:', venue_score)
print('QWK on all train rows with test-venue thresholds:', venue_score_all)
print('Distribution with test-venue thresholds:', pd.Series(venue_labels_all).value_counts().sort_index().to_dict())

USE_TEST_VENUE_THRESHOLDS = False
final_thresholds = venue_thresholds if USE_TEST_VENUE_THRESHOLDS else best['thresholds']
print('Using thresholds:', np.round(final_thresholds, 5))

## Create Submission

The submission concatenates public and private test rows, matching your existing SciBERT submissions.

In [ ]:
blend_test_pred = sum(best['weights'][i] * test_preds[name] for i, name in enumerate(model_names))
blend_test_labels = apply_thresholds(blend_test_pred, final_thresholds).astype(int)

submission = pd.DataFrame({
    'id': test['id'].values,
    'Label': blend_test_labels,
})

submission_path = SUBMISSION_DIR / 'tfidf_metadata_author_qwk_ensemble.csv'
submission.to_csv(submission_path, index=False)

print('Saved:', submission_path)
print('Shape:', submission.shape)
print('Distribution:', submission['Label'].value_counts().sort_index().to_dict())
display(submission.head())

## Save OOF and Test Continuous Predictions

These files are useful for later blending with SciBERT continuous predictions.

In [ ]:
oof_out = pd.DataFrame({
    'id': train['id'].values,
    'Label': y,
    'blend_pred': best['pred'],
    'blend_label': blend_oof_labels,
})
for name in model_names:
    oof_out[f'{name}_pred'] = oof_preds[name]

test_out = pd.DataFrame({
    'id': test['id'].values,
    'split': test['_split'].values,
    'blend_pred': blend_test_pred,
    'blend_label': blend_test_labels,
})
for name in model_names:
    test_out[f'{name}_pred'] = test_preds[name]

oof_path = SUBMISSION_DIR / 'oof_tfidf_metadata_author_qwk_ensemble.csv'
test_pred_path = SUBMISSION_DIR / 'test_pred_tfidf_metadata_author_qwk_ensemble.csv'

oof_out.to_csv(oof_path, index=False)
test_out.to_csv(test_pred_path, index=False)

print('Saved:', oof_path)
print('Saved:', test_pred_path)
display(oof_out.head())
display(test_out.head())

## Notes for SciBERT Blending

If your SciBERT notebook can save OOF/test continuous predictions, blend them with `blend_pred` from this notebook and optimize thresholds again. Good first weights to try:

- `0.60 * scibert + 0.40 * tfidf_metadata`,
- `0.50 * scibert + 0.50 * tfidf_metadata`,
- `0.40 * scibert + 0.60 * tfidf_metadata`.

Use OOF QWK to choose the weight, then apply the same weight to the test continuous predictions.